In [5]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


In [25]:
##########################################################
# Generic Training Function
##########################################################

def train_model(file_path, target_column, model_name):

    print(f"\nTraining {model_name}...")

    df = pd.read_csv(file_path)

    # Remove duplicates
    df = df.drop_duplicates()

    # Drop unwanted columns
    if 'Id' in df.columns:
        df = df.drop(columns=['Id'])

    if 'Num' in df.columns:
        df = df.drop(columns=['Num'])

    # Separate features and target
    X = df.drop(columns=[target_column])
    y = df[target_column]

    # Convert target if categorical
    if y.dtype == "object":
        le = LabelEncoder()
        y = le.fit_transform(y)

    # Identify column types
    numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
    categorical_cols = X.select_dtypes(include=['object']).columns

    # Numerical preprocessing
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    # Categorical preprocessing
    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    # Combine preprocessing
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numerical_cols),
            ('cat', categorical_transformer, categorical_cols)
        ]
    )

    # Random Forest model
    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )

    # Pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    # Train
    pipeline.fit(X_train, y_train)

    # Predict
    predictions = pipeline.predict(X_test)

    # Accuracy
    accuracy = accuracy_score(y_test, predictions)

    print(f"Accuracy: {accuracy:.4f}")

    print("\nClassification Report")
    print(classification_report(y_test, predictions))

    # Save model
    joblib.dump(pipeline, f"../backend/app/models/{model_name}.pkl")

    print(f"{model_name}.pkl saved successfully.")



In [26]:
##########################################################
# Train Heart Disease
##########################################################

train_model(
    "../datasets/heart.csv",
    "TargetBinary",
    "heart_model"
)




Training heart_model...
Accuracy: 0.8488

Classification Report
              precision    recall  f1-score   support

           0       0.86      0.86      0.86       111
           1       0.84      0.83      0.83        94

    accuracy                           0.85       205
   macro avg       0.85      0.85      0.85       205
weighted avg       0.85      0.85      0.85       205

heart_model.pkl saved successfully.


In [27]:
##########################################################
# Train Diabetes
##########################################################

train_model(
    "../datasets/diabetes.csv",
    "Outcome",
    "diabetes_model"
)




Training diabetes_model...
Accuracy: 0.7468

Classification Report
              precision    recall  f1-score   support

           0       0.79      0.83      0.81       100
           1       0.65      0.59      0.62        54

    accuracy                           0.75       154
   macro avg       0.72      0.71      0.72       154
weighted avg       0.74      0.75      0.74       154

diabetes_model.pkl saved successfully.


In [28]:
##########################################################
# Train Kidney Disease
##########################################################

train_model(
    "../datasets/kidney.csv",
    "Class",
    "kidney_model"
)




Training kidney_model...
Accuracy: 1.0000

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        30
           1       1.00      1.00      1.00        50

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80

kidney_model.pkl saved successfully.


In [29]:
##########################################################
# Train Stroke
##########################################################

train_model(
    "../datasets/stroke.csv",
    "Stroke",
    "stroke_model"
)

print("\nAll models trained successfully!")


Training stroke_model...
Accuracy: 0.9481

Classification Report
              precision    recall  f1-score   support

           0       0.95      0.99      0.97       972
           1       0.29      0.04      0.07        50

    accuracy                           0.95      1022
   macro avg       0.62      0.52      0.52      1022
weighted avg       0.92      0.95      0.93      1022

stroke_model.pkl saved successfully.

All models trained successfully!
